# The lid-driven cavity as Reynolds number rises

**Book:** §6.5.3, Figure 6.16 &nbsp;·&nbsp; `ch06/cavity_reynolds.ipynb`

The same vanilla PINN of §6.5.2, at **Re = 100, 400, and 1000**, against a streamfunction–vorticity
FD reference that is validated against Ghia *et al.* at each Re before it is used to judge the
network.

**The result is the honest frontier of the method.** The FD reference stays within ~2% of Ghia at
every Re. The identical PINN goes from ~10% (poor but recognisable) at Re = 100, to **72%** at
Re = 400, to **91%** at Re = 1000 — and, tellingly, its centreline minimum velocity moves the
*wrong way*: the real primary vortex deepens with Re, while the PINN's grows shallower. As the
range of scales widens, a single smooth network can no longer resolve the thin wall layers and the
strengthening corner vortices, and spectral bias (§5.2.1) collides head-on with the physics.

This is why the book's capstone is a cavity at Re = 100, and why turbulence (Chapter 7) is out of
reach. Warning: three cavity trainings — the most expensive computation in the kit. Use a GPU.

In [ ]:
"""Lid-driven cavity at Re = 100, 400, 1000, with an identical vanilla PINN at each.
FD reference (streamfunction-vorticity, GPU) validated against Ghia et al. before use.
The point: the PINN error grows sharply with Re, while the FD stays accurate. Honest demo."""
import time, json, numpy as np, torch, torch.nn as nn, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt

dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", dev)
torch.manual_seed(0); np.random.seed(0)

# Ghia et al. (1982): min of u on the vertical centreline -- our validation target
GHIA_UMIN = {100: -0.2058, 400: -0.3273, 1000: -0.3829}

# ------------------------------------------------------------------ FD reference (GPU)
def cavity_fd(RE, n=129, max_iter=120000, tol=2e-7, poisson=40):
    h = 1.0/(n-1)
    psi = torch.zeros((n, n), device=dev); om = torch.zeros((n, n), device=dev)
    dt = min(0.24*h*h*RE, 0.4*h)
    t0 = time.perf_counter()
    for it in range(max_iter):
        for _ in range(poisson):
            psi[1:-1,1:-1] = 0.25*(psi[2:,1:-1]+psi[:-2,1:-1]+psi[1:-1,2:]+psi[1:-1,:-2]
                                   + h*h*om[1:-1,1:-1])
        u = torch.zeros((n,n), device=dev); v = torch.zeros((n,n), device=dev)
        u[1:-1,1:-1] = (psi[1:-1,2:]-psi[1:-1,:-2])/(2*h)
        v[1:-1,1:-1] = -(psi[2:,1:-1]-psi[:-2,1:-1])/(2*h)
        u[:,-1] = 1.0
        om_new = om.clone()
        om_new[1:-1,0]  = -2*psi[1:-1,1]/h**2
        om_new[1:-1,-1] = -2*psi[1:-1,-2]/h**2 - 2.0/h
        om_new[0,1:-1]  = -2*psi[1,1:-1]/h**2
        om_new[-1,1:-1] = -2*psi[-2,1:-1]/h**2
        adv = (u[1:-1,1:-1]*(om[2:,1:-1]-om[:-2,1:-1])/(2*h)
               + v[1:-1,1:-1]*(om[1:-1,2:]-om[1:-1,:-2])/(2*h))
        lap = (om[2:,1:-1]+om[:-2,1:-1]+om[1:-1,2:]+om[1:-1,:-2]-4*om[1:-1,1:-1])/h**2
        om_new[1:-1,1:-1] = om[1:-1,1:-1] + dt*(lap/RE - adv)
        if it % 500 == 0:
            d = (om_new-om).abs().max()/(om_new.abs().max()+1e-12)
            if d < tol and it > 2000:
                break
        om = om_new
    if dev.type=='cuda': torch.cuda.synchronize()
    g = np.linspace(0,1,n)
    uc = u[n//2,:].cpu().numpy(); vc = v[:,n//2].cpu().numpy()
    U = u.cpu().numpy(); V = v.cpu().numpy()
    print(f"  FD  Re={RE:5.0f}: it={it:6d}  u_min={uc.min():.4f}  (Ghia {GHIA_UMIN[RE]:+.4f})  "
          f"[{time.perf_counter()-t0:.0f}s]")
    return g, uc, vc, U, V

# ------------------------------------------------------------------ vanilla PINN
def cavity_pinn(RE, epochs=25000, seed=0):
    torch.manual_seed(seed)
    net = nn.Sequential(nn.Linear(2,96), nn.Tanh(), nn.Linear(96,96), nn.Tanh(),
                        nn.Linear(96,96), nn.Tanh(), nn.Linear(96,96), nn.Tanh(),
                        nn.Linear(96,3)).to(dev)
    def uvp(x,y):
        o = net(torch.cat([x,y],1)); return o[:,0:1], o[:,1:2], o[:,2:3]
    def grads(f,*xs):
        return [torch.autograd.grad(f,x,torch.ones_like(f),create_graph=True)[0] for x in xs]
    opt = torch.optim.Adam(net.parameters(), 1e-3)
    z2 = torch.zeros(1,1,device=dev)
    t0 = time.perf_counter()
    for e in range(epochs):
        if e == int(0.60*epochs):
            for gr in opt.param_groups: gr['lr'] = 3e-4
        if e == int(0.84*epochs):
            for gr in opt.param_groups: gr['lr'] = 8e-5
        opt.zero_grad()
        N = 4096
        x = torch.rand(N,1,device=dev).requires_grad_(True)
        y = torch.rand(N,1,device=dev).requires_grad_(True)
        u,v,p = uvp(x,y)
        ux,uy = grads(u,x,y); vx,vy = grads(v,x,y); px,py = grads(p,x,y)
        uxx = grads(ux,x)[0]; uyy = grads(uy,y)[0]; vxx = grads(vx,x)[0]; vyy = grads(vy,y)[0]
        rx = u*ux + v*uy + px - (uxx+uyy)/RE
        ry = u*vx + v*vy + py - (vxx+vyy)/RE
        rc = ux + vy
        s = torch.rand(512,1,device=dev)
        wx = torch.cat([s, s, torch.zeros_like(s), torch.ones_like(s)])
        wy = torch.cat([torch.zeros_like(s), torch.ones_like(s), s, s])
        uw,vw,_ = uvp(wx,wy)
        lid = (wy > 0.999).float()
        bc = ((uw-lid)**2).mean() + (vw**2).mean()
        _,_,p0 = uvp(z2,z2)
        loss = (rx**2).mean() + (ry**2).mean() + 2*(rc**2).mean() + 10*bc + p0[0,0]**2
        loss.backward(); opt.step()
    if dev.type=='cuda': torch.cuda.synchronize()
    dt = time.perf_counter()-t0
    return net, uvp, dt

def centreline(uvp, g):
    yt = torch.tensor(g, dtype=torch.float32, device=dev).reshape(-1,1)
    half = torch.full_like(yt, 0.5)
    with torch.no_grad():
        uc = uvp(half, yt)[0].cpu().numpy().ravel()
        vc = uvp(yt, half)[1].cpu().numpy().ravel()
    return uc, vc

# ------------------------------------------------------------------ run the sweep
RES = [100.0, 400.0, 1000.0]
M = {}
data = {}
for RE in RES:
    g, u_fd, v_fd, U_fd, V_fd = cavity_fd(RE)
    net, uvp, dt = cavity_pinn(RE)
    u_pn, v_pn = centreline(uvp, g)
    eu = float(np.sqrt(np.mean((u_pn-u_fd)**2)/np.mean(u_fd**2)))
    ev = float(np.sqrt(np.mean((v_pn-v_fd)**2)/np.mean(v_fd**2 + 1e-9)))
    M[int(RE)] = dict(err_u=eu, err_v=ev, pinn_umin=float(u_pn.min()),
                      fd_umin=float(u_fd.min()), ghia_umin=GHIA_UMIN[int(RE)], train_s=dt)
    data[int(RE)] = dict(g=g, u_fd=u_fd, u_pn=u_pn, v_fd=v_fd, v_pn=v_pn)
    print(f"  PINN Re={RE:5.0f}: centreline rel L2  u={eu:.3f}  v={ev:.3f}  "
          f"u_min={u_pn.min():.4f}  [{dt:.0f}s]\n")

# ------------------------------------------------------------------ figure
fig, ax = plt.subplots(1, len(RES)+1, figsize=(4.0*(len(RES)+1), 4.2))
cols = ['tab:blue','tab:orange','tab:red']
for k,(RE,c) in enumerate(zip(RES, cols)):
    d = data[int(RE)]
    ax[k].plot(d['u_fd'], d['g'], 'g', lw=2.4, alpha=.65, label='FD $129^2$ ($\\approx$ Ghia)')
    ax[k].plot(d['u_pn'], d['g'], 'r--', lw=1.7,
               label=f"PINN (rel $L_2$={M[int(RE)]['err_u']:.2f})")
    ax[k].set_xlabel('$u(0.5,\\,y)$'); ax[k].set_ylabel('y'); ax[k].grid(alpha=.3)
    ax[k].legend(fontsize=8.5, loc='upper left')
    ax[k].set_title(f"$Re={int(RE)}$", fontsize=11)
# summary panel: error vs Re
res = [int(r) for r in RES]
eu = [M[r]['err_u']*100 for r in res]
ax[-1].plot(res, eu, 'o-', color='tab:red', lw=2, ms=8)
for r,e in zip(res,eu): ax[-1].annotate(f'{e:.0f}%', (r,e), textcoords='offset points',
                                        xytext=(6,6), fontsize=9)
ax[-1].set_xscale('log'); ax[-1].set_xlabel('Reynolds number'); ax[-1].set_ylabel('centreline error (%)')
ax[-1].grid(alpha=.3); ax[-1].set_title('The error grows with $Re$', fontsize=11)
ax[-1].set_xticks(res); ax[-1].set_xticklabels([str(r) for r in res])
plt.tight_layout(); plt.show()
print("METRICS", json.dumps(M, indent=1))
